# Module `dynamic_costs.py`

Ce notebook explique la logique de cout dynamique appliquee a chaque vraie arete du graphe residuel.

Important : on modifie toujours les vraies aretes. La matrice completee par Dijkstra est ensuite reconstruite par `DynamicNetworkSimulator`, pas modifiee directement. Ce decouplage garantit la coherence de la fermeture metrique.

## 1. Les trois niveaux de cout

Chaque arete a trois couts qui coexistent :

| Niveau | Definition | Calcul |
|---|---|---|
| `base_cost` | Distance physique minimale | `math.dist(p_u, p_v)` |
| `static_cost` | Cout apres surcout statique | `base_cost * (1 + static_surcharge)` ou `inf` si FORBIDDEN |
| dynamic cost | Cout courant a un tour donne | Gaussienne recentree autour de `static_cost`, bornee en bas par `base_cost` et en haut par `static_cost * max_multiplier` |

La dynamique respecte donc deux garde-fous :

- elle ne peut jamais descendre sous le cout physique minimal (une route ne peut pas etre plus rapide que sa distance reelle) ;
- elle ne peut jamais depasser un multiple configurable du cout statique (securite contre les derives).

In [ ]:
import sys
from pathlib import Path
import random

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.dynamic_costs import (
    DEFAULT_DYNAMIC_SIGMA,
    dynamic_multiplier,
    initialize_dynamic_edge_costs,
    refresh_dynamic_edge_costs,
    sample_dynamic_edge_cost,
)
from cesipath.models import EdgeAttributes, EdgeStatus

## 2. La formule en detail

`sample_dynamic_edge_cost(edge, previous_cost, sigma, mean_reversion_strength, max_multiplier)` applique :

```text
baseline      = previous_cost si defini sinon static_cost
reverted_mean = baseline - mean_reversion_strength * (baseline - static_cost)

relative_gap      = max(0, baseline/static_cost - 1)
volatility_scale  = max(0.35, 1 - 0.5 * min(relative_gap, 1))
std_deviation     = max(static_cost * sigma * volatility_scale, 0.01)

sample = gauss(reverted_mean, std_deviation)
return min(static_cost * max_multiplier, max(base_cost, sample))
```

Traduction intuitive :

- **Retour vers la moyenne** : plus le cout precedent s'eloigne du cout statique, plus le centre du tirage suivant est tire vers le cout statique.
- **Volatilite amortie** : quand le cout est deja tres au-dessus du statique, la variance diminue pour eviter de partir encore plus haut.
- **Bornes dures** : le resultat est tronque entre le cout physique et `static_cost * max_multiplier`.

In [ ]:
edge = EdgeAttributes(base_cost=10.0, status=EdgeStatus.SURCHARGED, static_surcharge=0.25)
print("base_cost   :", edge.base_cost)
print("static_cost :", edge.static_cost)
print("max autorise :", edge.static_cost * 1.8)

## 3. Evolution sur 100 tours (plot)

On simule une seule arete sur 100 tours. Le trace montre le retour vers `static_cost` quand le bruit s'eloigne, et le plafond `static_cost * max_multiplier`.

In [ ]:
import matplotlib.pyplot as plt

rng = random.Random(42)
history = [edge.static_cost]
current = edge.static_cost
for _ in range(100):
    current = sample_dynamic_edge_cost(
        edge,
        previous_cost=current,
        rng=rng,
        sigma=0.18,
        mean_reversion_strength=0.35,
        max_multiplier=1.8,
    )
    history.append(current)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history, label="cout dynamique", color="#3498db")
ax.axhline(edge.base_cost, color="#2ecc71", linestyle="--", label="base_cost (plancher)")
ax.axhline(edge.static_cost, color="#9b59b6", linestyle="--", label="static_cost (centre du retour)")
ax.axhline(edge.static_cost * 1.8, color="#e74c3c", linestyle="--", label="plafond (x 1.8)")
ax.set_xlabel("tour")
ax.set_ylabel("cout")
ax.set_title("Evolution d'un cout dynamique sur 100 tours")
ax.legend(loc="best")
ax.grid(True, linestyle=":", alpha=0.5)
plt.show()

## 4. `dynamic_multiplier` : lecture du coefficient

Donne `dynamic_cost / static_cost`. Utile pour trier les aretes les plus surchargees ou pour afficher une jauge dans une interface.

In [ ]:
for offset in [-2, 0, 2, 5]:
    cost = edge.static_cost + offset
    print(f"cout={cost:>6.2f}  coefficient={dynamic_multiplier(edge, cost)}")

## 5. `initialize_dynamic_edge_costs` et `refresh_dynamic_edge_costs`

Deux helpers pour travailler sur tout un dict d'aretes :

- `initialize_dynamic_edge_costs(edges)` : fixe chaque cout dynamique au `static_cost` correspondant (les FORBIDDEN sont ignorees).
- `refresh_dynamic_edge_costs(edges, previous_costs, rng, ...)` : applique `sample_dynamic_edge_cost` a chaque arete non interdite en utilisant le cout precedent comme `baseline`.

Dans le pipeline complet, `DynamicNetworkSimulator.advance()` utilise directement `sample_dynamic_edge_cost` par arete afin de pouvoir entrelacer l'echantillonnage et les tests de validation. Les deux helpers restent utiles en debug et dans les tests manuels.

In [ ]:
edges = {
    (0, 1): EdgeAttributes(base_cost=10.0, status=EdgeStatus.SURCHARGED, static_surcharge=0.25),
    (1, 2): EdgeAttributes(base_cost=8.0),
    (0, 2): EdgeAttributes(base_cost=6.0, status=EdgeStatus.FORBIDDEN),
}

network_rng = random.Random(7)
current_costs = initialize_dynamic_edge_costs(edges)
print("Etat initial (FORBIDDEN exclue) :", current_costs)

current_costs = refresh_dynamic_edge_costs(
    edges,
    current_costs,
    rng=network_rng,
    sigma=0.18,
    mean_reversion_strength=0.35,
    max_multiplier=1.8,
)
print("Apres un tour                   :", current_costs)

## 6. Pourquoi ce modele ?

Trois objectifs conjoints :

- **Realisme** : les couts oscillent de facon continue, comme un trafic routier reel.
- **Securite mathematique** : les bornes dures empechent les solveurs de voir des valeurs aberrantes.
- **Controle pedagogique** : les trois parametres (`sigma`, `mean_reversion_strength`, `max_multiplier`) sont intuitifs et documentes, ce qui permet de comparer aisement deux reglages dans les benchmarks dynamiques.